In [20]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import minimize, least_squares

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

sns.set_style('ticks', rc={"axes.facecolor": (0, 0, 0, 0)})
sns.set_context('talk')

from matplotlib import rcParams
rcParams['font.family'] = 'DejaVu Sans'

### Settings and input data

In [21]:
# ---- Settings ---- #
np.random.seed(100)

# ---- Input and output files ---- #
growth = Path("../../data/For_statistical_analysis_susceptible_strains.xlsx")

out_params = Path("../../out/Growth_features_estimates.tsv")
out_failed = Path("../../out/Growth_features_estimates_failed_samples.tsv")

# ---- prepare output files (headers) ---- #
out_params.write_text("\t".join(["ID","Method","L","r","t0","L0","H","lag"]) + "\n")
out_failed.write_text("ID\n")

3

In [22]:
# ---- read inputs ---- #
# read data and format
gd = pd.read_excel(growth, index_col=0)
gd.index.name = 'ID_FINAL'
gd_l = gd.reset_index().melt(id_vars='ID_FINAL', var_name='Hours', value_name='OD')
gd_l['Hours'] = gd_l['Hours'].astype(float)

In [23]:
# get sample and replicate
gd_l[['Sample', 'Replicate']] = gd_long['ID_FINAL'].str.rsplit('.', n=1, expand=True)

In [24]:
gd_l

,ID_FINAL,Hours,OD,Sample,Replicate
0,D39_1.1,0.0,0.004500,D39_1,1
1,D39_1.2,0.0,0.006500,D39_1,2
2,D39_dHCS_1.1,0.0,0.005000,D39_dHCS_1,1
3,D39_dHCS_1.2,0.0,0.012000,D39_dHCS_1,2
4,D39_16Pro_1.1,0.0,0.009625,D39_16Pro_1,1
...,...,...,...,...,...
29695,ATCC17619_4Pro_3.2,22.0,0.244250,ATCC17619_4Pro_3,2
29696,ATCC17619_4IgG_3.1,22.0,0.239875,ATCC17619_4IgG_3,1
29697,ATCC17619_4IgG_3.2,22.0,0.235875,ATCC17619_4IgG_3,2
29698,ATCC17619_4IgG_dHCS_3.1,22.0,0.346125,ATCC17619_4IgG_dHCS_3,1


### Define functions

In [25]:
# ---- functions ---- #
# biological function
def lgf(L, r, t0, L0, t):
    """4-parameter logistic function : L0 + (L / (1 + exp(-r*(t - t0))))"""
    t = np.asarray(t)
    return L0 + (L / (1.0 + np.exp(-r * (t - t0))))

# optimization functions
def residuals_sumsq(par, hours, od):
    """Objective for optimizers: sum of squared residuals"""
    pred = lgf(par[0], par[1], par[2], par[3], hours)
    return np.sum((od - pred) ** 2)

def residual_vector(par, hours, od):
    """Residual vector for least_squares (returns obs - pred)"""
    pred = lgf(par[0], par[1], par[2], par[3], hours)
    return od - pred

### Get parameters

In [34]:
gd_l

,ID_FINAL,Hours,OD,Sample,Replicate
0,D39_1.1,0.0,0.004500,D39_1,1
1,D39_1.2,0.0,0.006500,D39_1,2
2,D39_dHCS_1.1,0.0,0.005000,D39_dHCS_1,1
3,D39_dHCS_1.2,0.0,0.012000,D39_dHCS_1,2
4,D39_16Pro_1.1,0.0,0.009625,D39_16Pro_1,1
...,...,...,...,...,...
29695,ATCC17619_4Pro_3.2,22.0,0.244250,ATCC17619_4Pro_3,2
29696,ATCC17619_4IgG_3.1,22.0,0.239875,ATCC17619_4IgG_3,1
29697,ATCC17619_4IgG_3.2,22.0,0.235875,ATCC17619_4IgG_3,2
29698,ATCC17619_4IgG_dHCS_3.1,22.0,0.346125,ATCC17619_4IgG_dHCS_3,1


In [ ]:
# ---- main loop across SAMPLEs ----
samples = [s for s in gd_l["Sample"].unique() if s != "Blanc"]

for sample_id in tqdm(samples, desc="samples"): 
    sel_sample_all = gd_l[gd_l["Sample"] == sample_id].sort_values("Hours")
    reps = sel_sample_all["ID_FINAL"].unique()

    # ---- fit models  ---- #
    for idx, rep in enumerate(reps):
        sel_rep = sel_sample_all[sel_sample_all["ID_FINAL"] == rep].sort_values("Hours")
        hours = sel_rep["Hours"].values.astype(float)
        od = sel_rep["OD"].values.astype(float)
        if len(od) == 0:
            with out_failed.open("a") as fh:
                fh.write(f"{sample_id}.{rep}\n")
            continue

        max_idx = np.nanargmax(od)
        Max_time = hours[max_idx]
        Min_time = 0.0
        mask_main = (od <= od.max()) & (hours <= Max_time) & (hours >= Min_time)
        mask_tail = (np.abs(od - od.max()) / od.max() <= 0.05) & (hours > Max_time)
        keep_mask = mask_main | mask_tail

        fit_hours = hours[keep_mask] if keep_mask.sum() >= 4 else hours
        fit_od = od[keep_mask] if keep_mask.sum() >= 4 else od

        x0 = np.array([1.0, 1.0, 1.0, 1.0])
        bounds = [(0.1, 1.0), (1.0, 4.0), (0.0, 10.0), (0.0, 1.0)]
        methods = ["BFGS", "Nelder-Mead", "CG", "Powell", "L-BFGS-B"]

        # optimize minimizing SSR and pick lowest
        best_res = None
        best_ss = np.inf
        best_method = None
        for method in methods:
            try:
                if method == "L-BFGS-B":
                    res = minimize(lambda p: residuals_sumsq(p, fit_hours, fit_od),
                                   x0, method=method, bounds=bounds, options={"maxiter":20000})
                else:
                    res = minimize(lambda p: residuals_sumsq(p, fit_hours, fit_od),
                                   x0, method=method, options={"maxiter":20000})
                if res.success:
                    par = res.x
                    if par[1] > 0 and lgf(par[0], par[1], par[2], par[3], 40) < 2.0:
                        ss = residuals_sumsq(par, fit_hours, fit_od)
                        if ss < best_ss:
                            best_ss = ss
                            best_res = res
                            best_method = method
            except Exception:
                continue

        if best_res is None:
            with out_failed.open("a") as fh:
                fh.write(f"{sample_id}.{rep}\n")
            continue

        # refine with least_squares
        lower = [b[0] for b in bounds]
        upper = [b[1] for b in bounds]
        try:
            lsq = least_squares(lambda p: residual_vector(p, fit_hours, fit_od),
                                best_res.x, bounds=(lower, upper), method="trf",
                                max_nfev=20000, ftol=1e-12, xtol=1e-12)
            par_final = lsq.x
        except Exception:
            par_final = best_res.x

        dense_t = np.linspace(fit_hours.min(), fit_hours.max(), 1000)
        dense_od = lgf(par_final[0], par_final[1], par_final[2], par_final[3], dense_t)
        threshold = 1.5 * fit_od.min()
        idxs = np.where(dense_od >= threshold)[0]
        lag_value = dense_t[idxs[0]] if len(idxs) > 0 else np.nan

        out_row = {
            "ID": f"{sample_id}.{rep}",
            "Method": best_method,
            "L": par_final[0],
            "r": par_final[1],
            "t0": par_final[2],
            "L0": par_final[3],
            "delta.H": par_final[0] + par_final[3] - fit_od.min(),
            "lag": lag_value
        }
        with out_params.open("a") as fh:
            fh.write("\t".join(str(out_row[k]) for k in ["ID","Method","L","r","t0","L0","delta.H","lag"]) + "\n")

print("Done. Outputs produced:")
print(f"- {out_params} ")
print(f"- {out_failed}")

samples:  83%|████████▎ | 275/330 [05:08<01:57,  2.13s/it]

In [35]:
gf = pd.read_csv("../../out/Growth_features_estimates_failed_samples.tsv", sep="\t")

In [32]:
ge = pd.read_csv("../../out/Growth_features_estimates.tsv", sep="\t")

In [36]:
gf

,ID
0,D39_1.D39_1
1,D39_dHCS_1.D39_dHCS_1
2,D39_16Pro_1.D39_16Pro_1
3,D39_16IgG_1.D39_16IgG_1
4,D39_16IgG_dHCS_1.D39_16IgG_dHCS_1
...,...
325,ATCC17619_8IgG_3.ATCC17619_8IgG_3
326,ATCC17619_8IgG_dHCS_3.ATCC17619_8IgG_dHCS_3
327,ATCC17619_4Pro_3.ATCC17619_4Pro_3
328,ATCC17619_4IgG_3. ATCC17619_4IgG_3


In [48]:
ge['sample'] = ge['ID'].str.split('.').str[0]
mean_per_sample = ge.groupby('sample')[['L', 'r', 't0', 'L0', 'H', 'lag']].mean().reset_index()

In [49]:
mean_per_sample

,sample,L,r,t0,L0,H,lag
0,PBCN0001,0.688547,2.566945,5.112879,0.112063,0.696190,4.075826
1,PBCN0006,0.655446,1.819249,5.742077,0.116304,0.664325,4.313925
2,PBCN0007,0.730320,2.491700,4.607425,0.116921,0.741192,3.485485
3,PBCN0008,0.760636,2.328500,5.370564,0.111910,0.767740,4.188077
4,PBCN0009,0.595768,2.462870,5.774181,0.111155,0.602926,4.763208
...,...,...,...,...,...,...,...
373,PBCN0489,0.755841,2.478660,5.929215,0.083520,0.765348,4.608525
374,PBCN0490,0.695837,2.636248,5.173926,0.085869,0.707637,3.934212
375,PBCN0491,0.718064,2.207721,5.575462,0.110932,0.725553,4.342259
376,PBCN0492,0.771711,2.255491,5.625311,0.081060,0.778495,4.214047
